# Olist E-Commerce Analytics: Customer, Sales & Operations Analysis

**Business context:** Olist is a Brazilian e-commerce marketplace connecting small businesses (sellers) with customers across major online marketplaces. This notebook analyzes ~100,000 orders placed between September 2016 and October 2018 to understand sales performance, customer behavior, product performance, seller performance, delivery operations, and customer satisfaction.

**Goal:** Identify concrete, data-backed opportunities for revenue growth, customer retention, and operational improvement.

**Tools used:** Python (NumPy, Pandas, Matplotlib, Seaborn) for data cleaning and analysis; SQL for business queries; Tableau for dashboards.


## Section 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)


## Section 2: Load Datasets

We load all eight raw Olist CSV files. No assumptions are made about their structure ahead of time -- everything is verified in Section 3.

In [ ]:
DATA_PATH = '../data/raw/'  

customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
orders = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
order_items = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
order_payments = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
order_reviews = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

print('customers:', customers.shape)
print('orders:', orders.shape)
print('order_items:', order_items.shape)
print('order_payments:', order_payments.shape)
print('order_reviews:', order_reviews.shape)
print('products:', products.shape)
print('sellers:', sellers.shape)
print('category_translation:', category_translation.shape)


## Section 3: Dataset Overview

### Data Model

The Olist dataset is a relational, star-schema-like structure centered on `orders`:

```
customers (1) ----- (many) orders (1) ----- (many) order_items ----- (many:1) products ----- (many:1) category_translation
                              |                        |
                              |                        +----- (many:1) sellers
                              |
                              +----- (many) order_payments
                              |
                              +----- (many) order_reviews
```

- `customers.customer_id` is a per-order customer key; `customer_unique_id` identifies the actual person across multiple orders. **This distinction matters a lot for customer analysis and is used throughout this notebook.**
- `orders.order_id` is the central fact table key, joined to items, payments, and reviews.
- `order_items.product_id` links to `products`, which links to `category_translation` for English category names.
- `order_items.seller_id` links to `sellers`.

### File-by-file summary

| File | Rows | Columns | Primary Key | Key Foreign Keys | Represents |
|---|---|---|---|---|---|
| customers | 99,441 | 5 | customer_id | - | One row per order-customer link |
| orders | 99,441 | 8 | order_id | customer_id | Order lifecycle timestamps & status |
| order_items | 112,650 | 7 | order_id + order_item_id | order_id, product_id, seller_id | Line items within an order |
| order_payments | 103,886 | 5 | order_id + payment_sequential | order_id | Payment method/installments/value |
| order_reviews | 99,224 | 7 | review_id | order_id | Customer review score & text |
| products | 32,951 | 9 | product_id | - | Product attributes & category |
| sellers | 3,095 | 4 | seller_id | - | Seller location |
| category_translation | 71 | 2 | product_category_name | - | PT -> EN category names |


In [ ]:
for name, df in [('customers', customers), ('orders', orders), ('order_items', order_items),
                  ('order_payments', order_payments), ('order_reviews', order_reviews),
                  ('products', products), ('sellers', sellers)]:
    print(f"--- {name} ---")
    df.info()
    print()


## Section 4: Data Quality Checks

We check for missing values, duplicates, invalid values, and referential integrity across tables before doing any analysis.

In [ ]:
print('Missing values per file:')
for name, df in [('customers', customers), ('orders', orders), ('order_items', order_items),
                  ('order_payments', order_payments), ('order_reviews', order_reviews),
                  ('products', products), ('sellers', sellers)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"{name}: {dict(nulls) if len(nulls) else 'no missing values'}")


**Findings:**
- `orders`: `order_approved_at` (160), `order_delivered_carrier_date` (1,783), and `order_delivered_customer_date` (2,965) have missing values. These are orders that never completed the full lifecycle (e.g. canceled/unavailable orders), which is expected and not an error.
- `products`: `product_category_name` and related description fields are missing for 610 products. We'll fill these with `'unknown'` rather than dropping them, since the order/revenue data for these products is still valid.
- `order_reviews`: `review_comment_title` and `review_comment_message` are frequently missing -- this is expected, since many customers leave a star rating without writing a comment.

In [ ]:
print('Duplicate rows per file:')
for name, df in [('customers', customers), ('orders', orders), ('order_items', order_items),
                  ('order_payments', order_payments), ('order_reviews', order_reviews),
                  ('products', products), ('sellers', sellers)]:
    print(f"{name}: {df.duplicated().sum()}")

print()
print('Order status breakdown:')
print(orders['order_status'].value_counts())


**Decision:** Only orders with `order_status == 'delivered'` (96,478 of 99,441 orders, ~97%) are used for revenue, delivery, and review analysis, since these represent completed transactions with a full, reliable timeline. Non-delivered orders (canceled, unavailable, still in transit, etc.) are excluded from revenue calculations to avoid overstating realized sales.

## Section 5: Data Cleaning

Cleaning decisions made (each one documented):
1. Convert all order/review date columns from string to `datetime`.
2. Fill missing `product_category_name` with `'unknown'` (610 products) rather than dropping them -- their order history is still valid.
3. Merge English category names from `category_translation`; where no translation exists, fall back to the original Portuguese name rather than losing the category label.
4. Filter to `order_status == 'delivered'` for all revenue/delivery/satisfaction analysis (documented above).
5. De-duplicate `order_reviews` to one review per order (a few orders have more than one review row over time) by keeping the most recent `review_answer_timestamp`.

In [ ]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

products['product_category_name'] = products['product_category_name'].fillna('unknown')
products = products.merge(category_translation, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna(products['product_category_name'])

delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f"Delivered orders: {len(delivered):,} of {len(orders):,} total orders ({len(delivered)/len(orders)*100:.1f}%)")

reviews_dedup = order_reviews.sort_values('review_answer_timestamp').drop_duplicates('order_id', keep='last')
print(f"Reviews after de-duplication: {len(reviews_dedup):,} (was {len(order_reviews):,})")


## Section 6: Data Transformation

We now build a single **order-level master table** by joining delivered orders with customers, item/revenue totals, delivery timing, and review scores. This master table is the base for almost all analysis that follows.

In [ ]:
# Revenue per order: sum of item price + freight across all items in the order
item_revenue = order_items.groupby('order_id').agg(
    items_price=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    n_items=('order_item_id', 'count')
).reset_index()
item_revenue['order_total'] = item_revenue['items_price'] + item_revenue['freight_value']

# Master table
master = delivered.merge(customers, on='customer_id', how='left')
master = master.merge(item_revenue, on='order_id', how='left')

# Delivery metrics
master['delivery_days'] = (master['order_delivered_customer_date'] - master['order_purchase_timestamp']).dt.days
master['delivery_delay_days'] = (master['order_delivered_customer_date'] - master['order_estimated_delivery_date']).dt.days
master['late_delivery'] = master['delivery_delay_days'] > 0
master['order_month'] = master['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Attach review score (one per order, after de-duplication)
master = master.merge(reviews_dedup[['order_id', 'review_score']], on='order_id', how='left')

print('Master table shape:', master.shape)
master[['order_id', 'customer_unique_id', 'order_total', 'delivery_days', 'late_delivery', 'review_score']].head()


## Section 7: Exploratory Analysis

### 7.1 Revenue & Sales Overview

In [ ]:
total_revenue = master['order_total'].sum()
total_orders = master['order_id'].nunique()
total_customers = master['customer_unique_id'].nunique()
avg_order_value = master['order_total'].mean()

print(f"Total revenue: R$ {total_revenue:,.2f}")
print(f"Total delivered orders: {total_orders:,}")
print(f"Unique customers: {total_customers:,}")
print(f"Average order value: R$ {avg_order_value:.2f}")


**Business Question:** How does revenue trend over time, and which months perform best?

**Finding:** Revenue grows steadily through 2017, peaking in November 2017 at ~R$1.15M -- consistent with Black Friday seasonality in Brazil.

**Business Implication:** Marketing spend, inventory, and staffing should be planned around this seasonal peak.

In [ ]:
monthly = master.groupby('order_month').agg(revenue=('order_total','sum'), orders=('order_id','nunique')).reset_index()
m = monthly[(monthly['order_month'] >= '2017-01') & (monthly['order_month'] <= '2018-08')]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(m['order_month'], m['revenue']/1000, marker='o', color='#2E5EAA', linewidth=2)
ax.set_title('Monthly Revenue Trend (Jan 2017 - Aug 2018)', fontweight='bold')
ax.set_ylabel('Revenue (R$ thousands)')
ax.set_xticks(range(0, len(m), 2))
ax.set_xticklabels(m['order_month'].iloc[::2], rotation=45, ha='right')
plt.tight_layout()
plt.show()


**Business Question:** Which product categories generate the most revenue?

**Finding:** Health & Beauty, Watches & Gifts, and Bed/Bath/Table are the top 3 categories. The top 10 categories account for **62.4%** of total revenue.

**Business Implication:** Revenue is concentrated -- these categories deserve priority for inventory and marketing investment, while the long tail of categories contributes marginally.

In [ ]:
item_cat = order_items.merge(products[['product_id','product_category_name_english']], on='product_id', how='left')
item_cat['product_category_name_english'] = item_cat['product_category_name_english'].fillna('unknown')
delivered_ids = set(master['order_id'])
item_cat_delivered = item_cat[item_cat['order_id'].isin(delivered_ids)].copy()
item_cat_delivered['revenue'] = item_cat_delivered['price'] + item_cat_delivered['freight_value']

category_revenue = item_cat_delivered.groupby('product_category_name_english').agg(
    revenue=('revenue','sum'), orders=('order_id','nunique')
).reset_index().sort_values('revenue', ascending=False)

top10_share = category_revenue.head(10)['revenue'].sum() / category_revenue['revenue'].sum() * 100
print(f"Top 10 categories account for {top10_share:.1f}% of revenue")

fig, ax = plt.subplots(figsize=(9, 5.5))
top10 = category_revenue.head(10).sort_values('revenue')
ax.barh(top10['product_category_name_english'], top10['revenue']/1000, color='#2E5EAA')
ax.set_title('Top 10 Product Categories by Revenue', fontweight='bold')
ax.set_xlabel('Revenue (R$ thousands)')
plt.tight_layout()
plt.show()


### 7.2 Customer Analytics

In [ ]:
customer_orders = master.groupby('customer_unique_id').agg(
    n_orders=('order_id','nunique'), total_spend=('order_total','sum')
).reset_index()

repeat_pct = (customer_orders['n_orders'] > 1).mean() * 100
print(f"Average orders per customer: {customer_orders['n_orders'].mean():.3f}")
print(f"Average customer spend: R$ {customer_orders['total_spend'].mean():.2f}")
print(f"Repeat customers: {repeat_pct:.2f}%  |  One-time customers: {100-repeat_pct:.2f}%")


**Finding:** Only about **3% of customers** place more than one order on this marketplace. This is a defining characteristic of the Olist dataset and shapes the retention/cohort analysis in the next section -- traditional high-frequency retention analysis is not very informative here, since the overwhelming majority of customers are naturally one-time buyers.

### 7.3 RFM Customer Segmentation

We segment customers using **Recency** (days since last purchase), **Frequency** (number of orders), and **Monetary** (total spend). Because ~97% of customers have Frequency = 1, we use a simple, explainable rule for Frequency scoring instead of quantile-based scoring (which would fail with so many tied values), while Recency and Monetary use quantile-based (1-5) scoring.

In [ ]:
snapshot_date = master['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = master.groupby('customer_unique_id').agg(
    Recency=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    Frequency=('order_id', 'nunique'),
    Monetary=('order_total', 'sum')
).reset_index()

rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

def f_score(f):
    if f == 1: return 1
    elif f == 2: return 3
    else: return 5
rfm['F_score'] = rfm['Frequency'].apply(f_score)

def segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 3 and m >= 4: return 'Champions'
    elif r >= 4 and f == 1 and m >= 4: return 'Big Spenders (New)'
    elif r >= 4 and m <= 2: return 'New / Low-Value Customers'
    elif r == 3: return 'Potential Loyalists'
    elif r <= 2 and f >= 3: return 'At Risk (Was Frequent)'
    elif r <= 2 and m >= 4: return 'At Risk (High Value)'
    elif r <= 2 and m <= 2: return 'Lost / Hibernating'
    else: return 'Needs Attention'

rfm['Segment'] = rfm.apply(segment, axis=1)

segment_summary = rfm.groupby('Segment').agg(
    customers=('customer_unique_id','count'), total_revenue=('Monetary','sum'), avg_spend=('Monetary','mean')
).reset_index().sort_values('total_revenue', ascending=False)
segment_summary['pct_customers'] = round(segment_summary['customers']/segment_summary['customers'].sum()*100, 1)
segment_summary['pct_revenue'] = round(segment_summary['total_revenue']/segment_summary['total_revenue'].sum()*100, 1)
segment_summary


**What the business should do with each segment:**
- **Big Spenders (New) / At Risk (High Value)** (~15% of customers, but **>55% of revenue combined**): The single highest-leverage group. Since repeat purchase is rare overall, the priority is converting their first great experience into a second purchase via targeted follow-up offers.
- **Champions** (1.1% of customers, the only segment with meaningfully repeat behavior, Frequency ~2.2): Reward with loyalty perks -- they are proof that repeat purchasing is possible on this platform.
- **Potential Loyalists** (20% of customers): Recently active, not yet high spend -- good candidates for a "welcome back" or cross-sell campaign.
- **Lost / Hibernating** (16.4% of customers, only 5.5% of revenue): Low-cost win-back email campaigns only; not worth heavy investment.

### 7.4 Cohort / Retention Analysis

In [ ]:
cohort_df = master[['customer_unique_id','order_id','order_purchase_timestamp']].copy()
cohort_df['order_month'] = cohort_df['order_purchase_timestamp'].dt.to_period('M')

first_purchase = cohort_df.groupby('customer_unique_id')['order_month'].min().reset_index()
first_purchase.columns = ['customer_unique_id', 'cohort_month']
cohort_df = cohort_df.merge(first_purchase, on='customer_unique_id')
cohort_df['period_number'] = (cohort_df['order_month'] - cohort_df['cohort_month']).apply(lambda x: x.n)

cohort_counts = cohort_df.groupby(['cohort_month','period_number'])['customer_unique_id'].nunique().reset_index()
cohort_pivot = cohort_counts.pivot(index='cohort_month', columns='period_number', values='customer_unique_id')
retention = cohort_pivot.divide(cohort_pivot.iloc[:,0], axis=0) * 100

fig, ax = plt.subplots(figsize=(11,7))
sns.heatmap(retention.iloc[:,1:7], annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=1, cbar_kws={'label':'% Retained'})
ax.set_title('Customer Retention by Acquisition Cohort', fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Acquisition Cohort (Month)')
plt.tight_layout()
plt.show()


**Important limitation (stated explicitly, as instructed):** Traditional cohort retention analysis assumes a meaningful share of customers return in later months. On this dataset, fewer than 2% of customers make any repeat purchase at all, so month-over-month retention rates are consistently under 1% for nearly every cohort. **This is not an error in the analysis -- it is a genuine, important finding about Olist's business model**: it behaves much more like a one-time-purchase marketplace than a subscription or habitual-repurchase business. The cohort heatmap is included for completeness and transparency, but the RFM segmentation (Section 7.3) is a more actionable framework for this dataset, since it does not depend on repeat-purchase behavior.

### 7.5 Product Analytics

In [ ]:
item_cat_delivered = item_cat_delivered.merge(master[['order_id','review_score']], on='order_id', how='left')

category_performance = item_cat_delivered.groupby('product_category_name_english').agg(
    orders=('order_id','nunique'), avg_review=('review_score','mean'), revenue=('price','sum')
).reset_index()
category_performance = category_performance[category_performance['orders'] >= 100]

worst_rated = category_performance.sort_values('avg_review').head(8)
print("Lowest-rated high-volume categories (100+ orders):")
print(worst_rated[['product_category_name_english','orders','avg_review']].to_string(index=False))


**Finding:** `office_furniture` has the lowest average review score (3.52) among categories with meaningful volume, noticeably below the marketplace average of 4.16.

**Business Implication:** This category is a concrete candidate for a quality/fulfillment review with its sellers.

### 7.6 Seller Analytics

In [ ]:
seller_performance = item_cat_delivered.groupby('seller_id').agg(
    orders=('order_id','nunique'), revenue=('price','sum'), avg_review=('review_score','mean')
).reset_index().sort_values('revenue', ascending=False)

n_sellers = seller_performance.shape[0]
top10pct_n = int(n_sellers * 0.10)
top10_share = seller_performance.head(top10pct_n)['revenue'].sum() / seller_performance['revenue'].sum() * 100
print(f"Total active sellers: {n_sellers:,}")
print(f"Top 10% of sellers ({top10pct_n}) generate {top10_share:.1f}% of marketplace revenue")


**Finding:** The top 10% of sellers (297 of 2,970) generate **67.1%** of total marketplace revenue -- a strong concentration.

**Business Implication:** The marketplace is highly dependent on a relatively small group of sellers. Retention programs, dedicated account management, and proactive support for this group would materially protect revenue.

### 7.7 Delivery & Logistics Analysis

In [ ]:
delivery_valid = master.dropna(subset=['delivery_days','delivery_delay_days'])
avg_delivery_days = delivery_valid['delivery_days'].mean()
late_rate = delivery_valid['late_delivery'].mean() * 100

print(f"Average delivery time: {avg_delivery_days:.2f} days")
print(f"Late delivery rate: {late_rate:.2f}%")

fig, ax = plt.subplots(figsize=(8,4.5))
d = delivery_valid[delivery_valid['delivery_days'] < 60]
sns.histplot(d['delivery_days'], bins=40, color='#2E5EAA', ax=ax)
ax.axvline(d['delivery_days'].mean(), color='#E8743B', linestyle='--', label=f"Mean: {d['delivery_days'].mean():.1f} days")
ax.set_title('Delivery Time Distribution', fontweight='bold')
ax.set_xlabel('Delivery Days (purchase to customer)')
ax.legend()
plt.tight_layout()
plt.show()


### 7.8 Customer Review Analysis

In [ ]:
review_by_delivery = master.dropna(subset=['review_score','late_delivery']).groupby('late_delivery')['review_score'].agg(['mean','count'])
print("Average review score by delivery status:")
print(review_by_delivery)

fig, ax = plt.subplots(figsize=(6,4.5))
vals = [review_by_delivery.loc[False,'mean'], review_by_delivery.loc[True,'mean']]
bars = ax.bar(['On-Time','Late'], vals, color=['#2E5EAA','#c0392b'])
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.08, f'{v:.2f}', ha='center', fontweight='bold')
ax.set_ylim(0,5)
ax.set_title('Avg Review Score: On-Time vs Late Delivery', fontweight='bold')
plt.tight_layout()
plt.show()


**Finding:** Orders delivered on time average a **4.29** review score; late orders average only **2.27** -- a 2-point drop.

**Important:** This is a strong *association*, not proof of causation. Other unmeasured factors could contribute. The correct framing is: **"Late deliveries are associated with substantially lower review scores"** -- not that lateness *causes* the drop, even though the pattern strongly suggests delivery reliability is one of the biggest levers available for improving customer satisfaction.

## Summary

This notebook covered data quality checks, cleaning, transformation, and exploratory analysis across revenue, customers, RFM segmentation, cohort/retention, products, sellers, delivery, and reviews. See the accompanying SQL file (`sql/ecommerce_analysis.sql`) for a complementary set of 20 business questions answered in pure SQL, and the final PDF report for the complete set of business insights and recommendations.